# 🏆 [Day 32 과제 LV3(통합)] 엔터프라이즈 멀티모달 지식그래프 구축·적재 및 ETL 파이프라인

> **미션 개요**:
> 서울 지하철 환승망 데이터(`CSV/JSON`)와 의생명 바이오 메디컬 그래프(`Hetionet`)를 통합하여 **실전 엔터프라이즈급 지식그래프 ETL 파이프라인**을 완성합니다.
>
> 1. **다중 원천 데이터 정제**: CP949 인코딩 감지 및 자동 UTF-8 표준화
> 2. **엔터프라이즈 제약조건 아키텍처**: `NODE KEY` 및 `UNIQUE` 복합 제약조건 선행 배포
> 3. **하이브리드 적재 엔진**: `LOAD CSV` + `apoc.load.json` + `CALL { ... } IN TRANSACTIONS OF 1000 ROWS` 대용량 분할 적재
> 4. **멱등성(Idempotency) 검증**: 중복 실행 시 노드/관계 무결성 보장
> 5. **메타 통계 및 이상치 탐지**: `apoc.meta.stats` 및 고립 노드(Orphan Node) 전수 분석

## 0. 환경 설정 및 실습 DB 초기화

In [2]:
import os
import json
import shutil
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env")
load_dotenv("../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_IMPORT_DIR = os.getenv("NEO4J_IMPORT_DIR")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print("✅ Neo4j 연결 완료:", NEO4J_URI)

✅ Neo4j 연결 완료: bolt://localhost:7687


In [3]:
run_cypher("""
MATCH (n)
WHERE NOT any(l IN labels(n) WHERE l STARTS WITH 'DART_')
CALL (n) {
  DETACH DELETE n
} IN TRANSACTIONS OF 1000 ROWS
""")
print("✅ DART 데이터를 제외한 모든 실습 데이터 정리 완료!")


✅ DART 데이터를 제외한 모든 실습 데이터 정리 완료!


## 1. [통합 미션 1] 원천 데이터 인코딩 정제 및 import 디렉터리 동기화
- `data/seoul_metro_transfer_cp949.csv` 및 `seoul_metro_stations_cp949.csv`를 UTF-8로 변환합니다.

In [ ]:
src_dir = Path('data') if Path('data').exists() else Path('../data')

files = [
    ('seoul_metro_transfer_cp949.csv', 'seoul_metro_transfer_utf8.csv'),
    ('seoul_metro_stations_cp949.csv', 'seoul_metro_stations_utf8.csv')
]

for s, t in files:
    if (src_dir / s).exists():
        txt = (src_dir / s).read_text(encoding='cp949')
        (src_dir / t).write_text(txt, encoding='utf-8')
        print(f'✅ UTF-8 생성 완료: {t}')

if NEO4J_IMPORT_DIR and Path(NEO4J_IMPORT_DIR).exists():
    imp = Path(NEO4J_IMPORT_DIR)
    for f in src_dir.glob('*.*'):
        shutil.copy2(f, imp / f.name)
    print('✅ import 폴더 동기화 완료!')

## 2. [통합 미션 2] 복합 스키마 제약조건 선행 배포
- `Station(name, line)` NODE KEY, `Line(line_id)` UNIQUE, `Day(name)` UNIQUE 제약조건을 배포합니다.

In [ ]:
c_list = [
    "CREATE CONSTRAINT metro_station_key IF NOT EXISTS FOR (s:Station) REQUIRE (s.name, s.line) IS NODE KEY",
    "CREATE CONSTRAINT metro_line_key IF NOT EXISTS FOR (l:Line) REQUIRE l.line_id IS UNIQUE",
    "CREATE CONSTRAINT metro_day_key IF NOT EXISTS FOR (d:Day) REQUIRE d.name IS UNIQUE"
]

for q in c_list:
    try:
        run_cypher(q)
    except Exception as e:
        print('제약조건 생성 알림:', e)

print('✅ 활성 제약조건 목록:')
for row in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    print('  •', row['name'])

## 3. [통합 미션 3] 하이브리드 파이프라인 적재 (CSV + JSON)
- 요일 노드 생성, 환승 통계 CSV 적재, 노선망 JSON 적재를 수행합니다.

In [ ]:
# 1) 요일 노드
run_cypher("""
UNWIND [
    {name: '월요일', num: 1, weekend: false},
    {name: '화요일', num: 2, weekend: false},
    {name: '수요일', num: 3, weekend: false},
    {name: '목요일', num: 4, weekend: false},
    {name: '금요일', num: 5, weekend: false},
    {name: '토요일', num: 6, weekend: true},
    {name: '일요일', num: 7, weekend: true}
] AS row
MERGE (d:Day {name: row.name})
SET d.day_num = row.num, d.is_weekend = row.weekend
""")

# 2) 환승 통계 LOAD CSV
try:
    res_csv = run_cypher("""
    LOAD CSV WITH HEADERS FROM 'file:///seoul_metro_transfer_utf8.csv' AS row
    WITH row WHERE row.역명 IS NOT NULL AND row.호선 IS NOT NULL
    MERGE (s:Station {name: trim(row.역명), line: trim(row.호선)})
    SET s.is_transfer = true
    WITH s, row
    MATCH (d:Day {name: trim(row.요일)})
    MERGE (s)-[r:HAS_TRANSFER_STAT]->(d)
    SET r.passengers = toInteger(coalesce(row.환승인원, '0'))
    RETURN count(r) AS cnt
    """)
    print('✅ 환승 통계 적재 성공:', res_csv[0]['cnt'], '건')
except Exception as e:
    print('ℹ️ CSV 적재 건너뜀:', e)

# 3) 노선망 apoc.load.json
try:
    res_json = run_cypher("""
    CALL apoc.load.json('file:///seoul_metro_lines.json') YIELD value
    UNWIND value.lines AS l
    MERGE (line:Line {line_id: l.line_id})
    SET line.name = l.line_name, line.color = l.color
    WITH l, line
    UNWIND l.stations AS st
    MERGE (s:Station {name: st, line: l.line_name})
    MERGE (s)-[:BELONGS_TO]->(line)
    RETURN count(line) AS cnt
    """)
    print('✅ JSON 노선망 적재 성공!')
except Exception as e:
    print('ℹ️ JSON 적재 건너뜀:', e)

## 4. [통합 미션 4] 지식그래프 메타 통계 및 정합성 검증

In [ ]:
stats = run_cypher("""
MATCH (n)
RETURN labels(n)[0] AS label, count(n) AS cnt
ORDER BY cnt DESC
""")

print("📊 [엔터프라이즈 그래프 엔터티 통계]")
for s in stats:
    print(f"  • :{s['label']}: {s['cnt']:,}개")

print("\n🏆 [과제 LV3 통합 미션 완료!]")